# 05 — Hubble Tension Fit (CI‑Aware)
**Abstract.**  
Compute the S.T.A.R. effective Hubble parameter \(H_{\mathrm{eff}}(z)\) and perform a simple fit/diagnostic comparing local distance indicators to CMB‑inferred values.

This CI‑aware version loads redshift–distance data from:
- A user‑provided cosmic dataset (if present)
- CI‑generated Cremona labels (synthesizing redshift–distance pairs)
- Synthetic fallback (local development)

Uses `src.physics.hubble_effective` and `src.physics.hubble_tension_fit` when available; otherwise uses a safe fallback linear model.


In [ ]:
import sys, os
sys.path.append('src')

import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
np.random.seed(11)

os.makedirs('results', exist_ok=True)


In [ ]:
# --- CI + arithmetic archive detection ---
RUNNING_IN_CI = os.environ.get("GITHUB_ACTIONS", "false") == "true"

CI_LABELS = "data/raw/ci_subset.csv"
EC_DATA = "external/ecdata"
LMFDB = "external/lmfdb"

print("Running in CI:", RUNNING_IN_CI)
print("CI labels:", os.path.exists(CI_LABELS))
print("Cremona ecdata:", os.path.exists(EC_DATA))
print("LMFDB:", os.path.exists(LMFDB))


In [ ]:
cosmic_path = 'data/raw/JApJ94494_2MASS_GAIADR3_EPOCH.csv'

if os.path.exists(cosmic_path):
    cosmic = pd.read_csv(cosmic_path)
    print("Loaded cosmic dataset:", cosmic_path)

    # Try to extract redshift or distance modulus columns
    if 'RAdeg' in cosmic.columns and 'DEdeg' in cosmic.columns:
        # No redshift column → synthesize z and distance
        z = np.clip(np.random.normal(loc=0.02, scale=0.01, size=len(cosmic)), 0.001, 0.2)
        dist = 10**((37.936 - 25)/5.0) * np.ones(len(cosmic))  # placeholder
    else:
        z = np.random.uniform(0.001, 0.2, size=200)
        dist = (3e5/70.0) * z * (1 + 0.5*z)

elif RUNNING_IN_CI and os.path.exists(CI_LABELS):
    print("CI mode: generating synthetic redshift–distance pairs from CI labels")
    labels = pd.read_csv(CI_LABELS)
    n = len(labels)

    z = np.sort(np.random.uniform(0.001, 0.2, size=n))
    H0_true = 70.0
    dist = (3e5 / H0_true) * z * (1 + 0.5*z) + np.random.normal(scale=5.0, size=n)

else:
    print("Cosmic dataset not found; generating synthetic redshift–distance pairs.")
    n = 120
    z = np.sort(np.random.uniform(0.001, 0.2, size=n))
    H0_true = 70.0
    dist = (3e5 / H0_true) * z * (1 + 0.5*z) + np.random.normal(scale=5.0, size=n)


In [ ]:
try:
    from src.physics.hubble_effective import HubbleEffective
    from src.physics.hubble_tension_fit import HubbleTensionFit

    HE = HubbleEffective()
    HT = HubbleTensionFit()

    # Build simple invariants and entropy_curvature arrays (synthetic)
    invariants_list = [{'omega': 1.0, 'rank': np.random.randint(1,3)} for _ in z]
    entropy_curvatures = np.random.normal(scale=0.1, size=len(z))

    H_eff_vals = np.array([
        HE.H_eff(zi, inv, ec)
        for zi, inv, ec in zip(z, invariants_list, entropy_curvatures)
    ])

    # Observed H from distances: H_obs = (c*z)/d
    H_obs = (3e5 * z) / (dist + 1e-12)

    residuals = H_eff_vals - H_obs
    chi2 = HT.chi_squared(residuals, sigma=np.maximum(1.0, 0.05*H_obs))

    print("Computed H_eff and residuals. chi2:", chi2)

except Exception as e:
    print("Hubble modules not available; using fallback linear H(z) model.", e)

    H_eff_vals = 70.0 * np.ones_like(z)
    H_obs = (3e5 * z) / (dist + 1e-12)
    residuals = H_eff_vals - H_obs
    chi2 = np.sum((residuals / np.maximum(1.0, 0.05*H_obs))**2)


In [ ]:
plt.figure(figsize=(7,5))
plt.errorbar(z, H_obs, yerr=0.05*H_obs, fmt='o', label='Observed (proxy)')
plt.plot(z, H_eff_vals, '-', lw=2, label='S.T.A.R. H_eff(z)')
plt.xlabel('z')
plt.ylabel('H (km/s/Mpc)')
plt.title('H_eff(z) vs Observed (proxy)')
plt.legend()
plt.grid(True)

plt.savefig('results/hubble_fit.png', dpi=200)
plt.show()

with open('results/hubble_fit_summary.json','w') as f:
    json.dump({
        'chi2': float(chi2),
        'mean_residual': float(np.mean(residuals))
    }, f)

print("Saved results/hubble_fit.png and results/hubble_fit_summary.json")


**Interpretation.**  
This notebook demonstrates how S.T.A.R. predictions for \(H_{\mathrm{eff}}(z)\) compare to local distance proxies.  
For rigorous validation, replace proxies with calibrated distance indicators (SN Ia, TRGB) and Planck CMB constraints.


In [ ]:
print("Notebook: 05_hubble_tension_fit")
print("Data used:",
      cosmic_path if os.path.exists(cosmic_path)
      else "CI labels" if RUNNING_IN_CI else "synthetic")
print("Outputs: results/hubble_fit.png, results/hubble_fit_summary.json")
